In [1]:
from transformers import BertTokenizer, BertModel
import torch
import os
import pandas as pd

/home/francia/anaconda3/envs/csr_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-large-cased')
model = BertModel.from_pretrained('bert-large-cased')
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [3]:
# path = '../CSR_report_processed_v4_gemini_v0/NASDAQ_AAL_2007_v0_gemini_corrected.txt'
# try:
#     with open(path, 'r') as file:
#         text = file.read()
# except:
#     print(f"Error: The file '{path}' was not found.")
#     exit(1)
# file.close()

In [4]:
max_len=512
chunk_size = max_len - 2
input_folder = "../CSR_report_processed_v4_gemini_v0"
output_csv = "../output_dataset/csr_embeddings.csv"

In [5]:
def get_embedding(text):
    # 兩個換行符號代表下一個段落
    paragraphs = [p.strip() for p in text.split(r'\n\s*\n') if p.strip()]
    embeddings = []
    for paragraph in paragraphs:
        tokens = tokenizer.tokenize(paragraph)
        chuncks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]

        for chunck in chuncks:
            tokens_chunck = ['[CLS]'] + chunck + ['[SEP]']
            input_ids = tokenizer.convert_tokens_to_ids(tokens_chunck)
            input_ids = torch.tensor([input_ids])

            with torch.no_grad():
                outputs = model(input_ids)
                last_hidden_states = outputs.last_hidden_state.squeeze(0)

            token_embeddings = last_hidden_states[1:-1]  # Exclude [CLS] and [SEP]
            chunck_embedding = token_embeddings.mean(dim=0)
            embeddings.append(chunck_embedding)
    doc_embedding = torch.stack(embeddings).mean(dim=0)
    return doc_embedding

In [6]:
if not os.path.exists(output_csv):
    df = pd.DataFrame(columns=["file_name"] + [f"dim_{i}" for i in range(1024)])
    df.to_csv(output_csv, index=False)

In [7]:
max_files = 1000

In [ ]:
processed_count = 0

for fname in sorted(os.listdir(input_folder)):
    if fname.endswith(".txt") and processed_count < max_files:
        try:
            full_path = os.path.join(input_folder, fname)
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()

            file_name = fname.split('_v0')[0]

            # 避免重複處理
            existing_df = pd.read_csv(output_csv)
            if file_name in existing_df["file_name"].values:
                print(f"✔️ Already processed: {file_name}")
                continue

            print(f"🚀 Processing: {file_name}")
            emb = get_embedding(text)

            # 儲存結果
            row = [file_name] + emb.tolist()
            df_new = pd.DataFrame([row], columns=["file_name"] + [f"dim_{i}" for i in range(1024)])
            df_new.to_csv(output_csv, mode='a', header=False, index=False)

            processed_count += 1

        except Exception as e:
            print(f"❌ Error processing {fname}: {e}")

✔️ Already processed: NASDAQ_AAL_2007
✔️ Already processed: NASDAQ_AAL_2008
✔️ Already processed: NASDAQ_AAL_2009
✔️ Already processed: NASDAQ_AAL_2011
✔️ Already processed: NASDAQ_AAL_2012
🚀 Processing: NASDAQ_AAL_2013
🚀 Processing: NASDAQ_AAL_2014
🚀 Processing: NASDAQ_AAL_2015
